# Student Acceptance and Interaction Patterns with Generative AI in Accessible UI Design
## Replication Package & Statistical Analysis (CLEI Electronic Journal)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

**Authors:** Daniela Marques (IFSP), André Luiz Satoshi Kawamoto (UTFPR), Valéria Farinazzo Martins (Mackenzie)

This notebook provides the complete reproducible pipeline for:
1. **UTAUT Pre-Post Analysis:** Parametric (Paired t-test) and Non-Parametric (Wilcoxon Signed-Rank) comparisons with Cohen's d effect sizes.
2. **Granular Item-Level Analysis:** Longitudinal evaluation of each questionnaire item.
3. **Subgroup Inferential Testing:** Mann-Whitney U tests for Novices vs. Experienced GenAI users; Kruskal-Wallis for UI experience.
4. **Prompt Engineering & UI Accessibility Audit:** Quantitative evaluation of prompt iteration counts and objective heuristic compliance in modified interfaces.

In [ ]:
# 1. Imports and Setup
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configure visual aesthetics for high-resolution scientific publishing
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["figure.dpi"] = 300

print("Libraries successfully imported.")

In [ ]:
# 2. Data Loading (Local or Zenodo/GitHub)
# In Google Colab, you can upload the files or fetch directly from GitHub/Zenodo:
url_paired = "https://raw.githubusercontent.com/alkawamo/clei-genai-study/main/zenodo_dataset/dataset_utaut_paired_pre_post.csv"
url_prompts = "https://raw.githubusercontent.com/alkawamo/clei-genai-study/main/zenodo_dataset/dataset_prompt_iterations_and_accessibility.csv"

# Fallback to local files if running locally:
local_paired = "../zenodo_dataset/dataset_utaut_paired_pre_post.csv"
local_prompts = "../zenodo_dataset/dataset_prompt_iterations_and_accessibility.csv"

if os.path.exists(local_paired):
    df_paired = pd.read_csv(local_paired)
    df_prompts = pd.read_csv(local_prompts)
else:
    df_paired = pd.read_csv(url_paired)
    df_prompts = pd.read_csv(url_prompts)

print(f"Loaded paired UTAUT dataset: {df_paired.shape[0]} participants, {df_paired.shape[1]} variables.")
print(f"Loaded prompt and accessibility dataset: {df_prompts.shape[0]} student projects audited.")

In [ ]:
# 3. Demographics and Sample Profiling (High-Resolution Figure for Paper)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Gender
sns.countplot(data=df_paired, x="gender", ax=axes[0], color="#3498db")
axes[0].set_title("(a) Gender Distribution", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Gender")
axes[0].set_ylabel("Number of Participants")

# UI Experience
exp_order = [
    "Nenhuma Experiência - Nunca desenvolvi nenhuma interface",
    "Pouca - Já implementei cerca de 1-5 interfaces simples",
    "Intermediário - Já implementei cerca de 1-5 interfaces com características mais complexas",
    "Avançado - Já implementei interfaces para mais de 10 sistemas, incluindo sistemas corporativos ou de grande escala"
]
short_labels = ["None", "Low (1-5)", "Intermediate", "Advanced (>10)"]
sns.countplot(data=df_paired, x="ui_experience", order=exp_order, ax=axes[1], color="#2ecc71")
axes[1].set_title("(b) UI Development Experience", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Experience Level")
axes[1].set_xticklabels(short_labels, rotation=20, ha="right")
axes[1].set_ylabel("Number of Participants")

# Prior AI Use
sns.countplot(data=df_paired, x="ai_used_before", ax=axes[2], color="#e67e22")
axes[2].set_title("(c) Prior Hands-on GenAI Use", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Used AI Before?")
axes[2].set_ylabel("Number of Participants")

plt.tight_layout()
plt.savefig("figure_demographics_clei.png", dpi=300)
plt.show()

In [ ]:
# 4. UTAUT Constructs: Paired t-Test, Wilcoxon Signed-Rank Test, and Cohen's d
constructs = ["PE", "EE", "SI", "FC", "BI"]
names = ["Performance Expectancy", "Effort Expectancy", "Social Influence", "Facilitating Conditions", "Behavioral Intention"]
results = []

for c, name in zip(constructs, names):
    pre = df_paired[f"{c}_pre"]
    post = df_paired[f"{c}_post"]
    
    # Parametric t-test
    t_stat, t_pval = stats.ttest_rel(post, pre)
    
    # Non-parametric Wilcoxon signed-rank test
    w_stat, w_pval = stats.wilcoxon(post, pre)
    
    # Cohen's d for paired samples: mean(diff) / sd(diff)
    diff = post - pre
    cohens_d = diff.mean() / diff.std(ddof=1) if diff.std(ddof=1) != 0 else 0.0
    
    results.append({
        "Construct": f"{name} ({c})",
        "Pre Mean (SD)": f"{pre.mean():.2f} ({pre.std():.2f})",
        "Post Mean (SD)": f"{post.mean():.2f} ({post.std():.2f})",
        "Delta (Post - Pre)": f"{diff.mean():+.2f}",
        "t-statistic": round(t_stat, 2),
        "t p-value": round(t_pval, 3),
        "Wilcoxon W": round(w_stat, 1),
        "Wilcoxon p-value": round(w_pval, 3),
        "Cohen's d": round(cohens_d, 2)
    })

df_utaut_summary = pd.DataFrame(results)
print("=== Paired Inferential Analysis of UTAUT Constructs (N = 41) ===")
display(df_utaut_summary)

In [ ]:
# 5. Subgroup Analysis: Novices vs. Experienced GenAI Users (Mann-Whitney U Test)
novices = df_paired[df_paired["ai_used_before"] == "Não"]
experienced = df_paired[df_paired["ai_used_before"] == "Sim"]

sub_results = []
for c in constructs:
    d_nov = novices[f"{c}_delta"]
    d_exp = experienced[f"{c}_delta"]
    u_stat, u_pval = stats.mannwhitneyu(d_nov, d_exp, alternative="two-sided")
    
    sub_results.append({
        "Construct": c,
        "Novices Mean Delta (n=5)": round(d_nov.mean(), 2),
        "Experienced Mean Delta (n=36)": round(d_exp.mean(), 2),
        "Mann-Whitney U": round(u_stat, 1),
        "p-value": round(u_pval, 3)
    })

df_subgroup_summary = pd.DataFrame(sub_results)
print("=== Subgroup Comparison: Shift in Acceptance (Delta) by Prior AI Experience ===")
display(df_subgroup_summary)

In [ ]:
# 6. Objective UI Accessibility Audit (Game Redesign Heuristics)
features = {
    "Font Size Scaling": "has_font_size_adjustment",
    "Pacing / Speed Control": "has_speed_reduction",
    "Multimodal Audio / Volume": "has_multimodal_audio",
    "Keyboard Navigation": "has_keyboard_navigation",
    "WAI-ARIA Attributes": "has_aria_accessibility_tags",
    "Contrast / Dark Theme": "has_contrast_theme_toggle",
    "Text Color Labels": "has_text_color_labels",
    "Explicit Round Feedback": "has_round_feedback"
}

feat_counts = {k: df_prompts[v].sum() for k, v in features.items()}
feat_pcts = {k: (df_prompts[v].sum() / len(df_prompts)) * 100 for k, v in features.items()}

df_feat = pd.DataFrame({"Feature": list(feat_counts.keys()), "Count": list(feat_counts.values()), "Percentage": list(feat_pcts.values())})
df_feat = df_feat.sort_values(by="Count", ascending=True)

plt.figure(figsize=(10, 5.5))
bars = plt.barh(df_feat["Feature"], df_feat["Percentage"], color="#16a085", edgecolor="black")
plt.xlabel("Percentage of Student Submissions Implementing Feature (%)", fontsize=11)
plt.title("Objective Accessibility Compliance of AI-Generated Interfaces (N = 26)", fontsize=13, fontweight="bold")
plt.xlim(0, 115)

for bar in bars:
    w = bar.get_width()
    plt.text(w + 2, bar.get_y() + bar.get_height()/2, f"{w:.1f}%", va="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("figure_accessibility_compliance_clei.png", dpi=300)
plt.show()

In [ ]:
# 7. Correlation Analysis: Prompt Attempts vs. Perception Shift
df_merged = pd.merge(df_paired, df_prompts[["participant_id", "prompt_iterations_count"]], on="participant_id")

corr_res = []
for c in constructs:
    rho, pval = stats.spearmanr(df_merged["prompt_iterations_count"], df_merged[f"{c}_delta"])
    corr_res.append({"Construct": c, "Spearman rho": round(rho, 3), "p-value": round(pval, 3)})

print("=== Correlation (Spearman) Between Prompt Iterations and Acceptance Shift (Delta) ===")
display(pd.DataFrame(corr_res))